# 🎯 [PRC01_M05] 계약 위험 점수 산출 및 규정 위반 가능성 신뢰도 지수 엔진 개발 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC01/prc01_m05_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC01_M05 |
| **미션 주제** | 계약 위험 점수 산출 및 규정 위반 가능성 신뢰도 지수 엔진 개발 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
조달계획안과 계약서는 지체상금률, 선금, 보증, 하도급, 필수 조항 같은 항목이 규정에 맞는지 사람이 한 건씩 읽고 대조해야 해서 검토가 늦어지고 불공정 조항이나 납기 지체 위험 문구를 놓치기 쉽다. 본 프로젝트는 조달계획안과 계약서의 오류와 위험을 검증하는 도구를 만드는 과제로, 조달 업무의 병목과 위험 신호 기준, 조항 수정 대안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 계약 자료로 계약 데이터 스키마 정의, 서식별 자료 정제와 조항 정규화, 룰 기반 규정 준수 판별, 자연어처리 기반 독소·납기 위험 문구 식별, 위험 점수와 신뢰도 엔진, 조항 하이라이트 대시보드와 오류 처리, 계약관·법무 검토 HITL 워크플로우, 1,000건 일괄 스크리닝 부하 시험, AAR 룰셋 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 계약을 근거와 함께 먼저 보여 주어 검토 시간을 줄이되, 규정 위반 여부의 판단과 계약 승인은 계약관·법무·승인권자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 📁 프로젝트 표준 산출물 디렉터리 체계 (`docs/`, `src/`, `dist/`) 안내
> **💡 [실무 프로토타입 업무 도구 구축 가이드]**
> 본 프로젝트는 단순 일회성 데이터 분석 실습이 아닌, 실제 군 부서(군수사령부, 조달본부, 계약관리과 등)에 배포 가능한 **업무 자동화 소프트웨어 도구(Tool)**를 단계별로 조립·완성하는 과정입니다. 실습 중 산출물(Deliverables)로 요구되는 파일들은 아래 표준 디렉터리에 모듈별로 저장·누적됩니다.
> - **`docs/` (국방 행정 및 기술 명세서):** 업무 정의서(`prc01_as_is_to_be_spec.md`), 스키마 명세서(`schema_definition.md`), UAT 검증 보고서(`prc01_uat_feedback_report.md`), 7개 장 운영 매뉴얼(`docs/prc01_final_manual.md`)
> - **`src/` (업무 도구 소스코드):** 데이터 파서(`connectors/`), 규정 준수 룰 엔진(`engine/`), NLP 지능 모듈(`ai/`), 대시보드 UI(`ui/`), 결함 검증기(`utils/`), HITL 워크플로우(`workflow/`), 배치 러너(`pipeline/`)
> - **`dist/` (배포 패키지):** 원클릭 자동 설치 스크립트(`dist/prc01_package/setup.sh`), 설치 가이드(`README.md`)
> *(주피터 노트북 환경에서는 각 미션 셀 검증 후 `%%writefile src/...` 매직 명령어를 활용하여 해당 모듈 파일로 자동 추출할 수 있습니다.)*

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 계약 위험 점수 산출 및 규정 위반 가능성 신뢰도 지수 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

계약 위험 점수 산출 및 규정 위반 가능성 신뢰도 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **룰 적중 내역과 조항 분류 결과를 결합해 계약 위험 점수와 규정 위반 가능성 신뢰도 지수를 산출하고 입력-분석-검토-결과 4개 화면을 구성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **룰 16종 적중 여부, 모델이 찾은 독소·납기위험 조항 수, 금액을 로지스틱 회귀로 결합해 0~100점의 위험 점수를 만들고, 점수의 뚜렷함과 룰·모델 일치 여부로 신뢰도 지수 정의하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/prc01_app.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무자 친화적 디자인 및 조작 편의성

#### 미션 PASS 조건

- 정량 기준: 핵심 화면 4종(입력-분석-검토-결과) 구현 완료


- 진행 기준: UI 프로토타입 로컬 서버 구동 및 전체 페이지 네비게이션 확인


- 검증 방법: 사용성 평가(UX Walkthrough) 및 인터랙션 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""룰 적중 내역과 조항 분류 모델의 위험 조항 수를 로지스틱 회귀로 결합해 계약별 위반 위험 점수(0~100)와 신뢰도 지수를 산출하는 엔진을 만들고, 별도 가상 계약군에서 성능·신뢰도 보정을 검증하며 입력-분석-검토-결과 4개 화면(텍스트 패널과 그래프)과 상태 뱃지·필터를 구성한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 조달계획안·계약서 생성기 (실제 계약·업체·법령 조문 없음)
# 계약 속성 16개 + 조항 문장 6개를 만들고, 속성 위반·독소 조항·납기 위험 문구가 섞인 '규정 위반 여부'를 함께 만든다.
# ---------------------------------------------------------------------------
TYPES = ["물품", "용역", "공사"]
METHODS = ["일반경쟁", "제한경쟁", "지명경쟁", "수의계약"]
CLASSES = ["정상", "독소", "납기위험"]
OBJ = ["납품 물품", "용역 결과물", "시공 목적물", "정비 성과물"]

N_TPL = [
    "계약상대자가 납기를 지체한 경우 지체일수 1일마다 계약금액의 {p}/1000에 해당하는 지체상금을 부과한다.",
    "지체상금은 불가항력 사유가 확인된 경우에 한하여 발주기관의 승인을 받아 면제할 수 있다.",
    "납기 연장은 발주기관이 불가항력 사유를 확인하고 서면으로 승인한 경우에만 인정한다.",
    "발주기관은 납품된 {o}을 {d}일 이내에 검사하고 합격한 후에 대금을 지급한다.",
    "계약상대자는 하도급을 하려는 경우 사전에 발주기관의 서면 승인을 받아야 한다.",
    "어느 한쪽이 계약을 해지하려면 사유를 명시해 {d}일 전에 서면으로 통지해야 한다.",
    "{o}에 하자가 발생하면 계약상대자는 보증 기간 안에 무상으로 보수하거나 교체한다.",
    "계약 이행 중 알게 된 비공개 사항은 제3자에게 누설하지 않으며 위반 시 책임을 진다.",
    "{o}에 대한 지식재산권의 귀속과 사용 범위는 별도 합의서에 따라 정한다.",
    "대금은 검사 합격 후 청구서를 접수한 날부터 {d}일 이내에 지급한다.",
    "계약금액의 조정은 물가변동 기준에 따라 양 당사자가 서면으로 합의한 경우에만 한다.",
    "계약상대자는 이행보증서를 계약 체결 시 제출하며 보증금은 계약 이행 완료 후 반환한다.",
    "납품 일정은 계약서 별표의 일자를 따르며 변경 시 서면 합의가 필요하다.",
    "지체상금은 면제 사유가 없는 한 부과하며 이의는 정해진 절차로 제기할 수 있다.",
    "검사에 합격하지 못한 {o}은 계약상대자가 교체하며 그 비용은 계약상대자가 부담한다.",
    "계약상대자의 귀책 사유로 이행이 불가능하면 발주기관은 서면으로 통지하고 계약을 해지할 수 있다.",
    "지연이 예상되면 계약상대자는 사유와 만회 계획을 발주기관에 서면으로 보고한다.",
    "보증 기간 중 발생한 결함은 계약상대자가 책임지고 보수한다.",
    "분쟁은 먼저 협의로 해결하고 협의가 되지 않으면 관할 법원의 판단에 따른다.",
]
T_TPL = [
    "계약상대자의 지체상금은 전액 면제하며 발주기관은 이에 대해 이의를 제기하지 못한다.",
    "납품 즉시 검사에 합격한 것으로 보며 발주기관은 이후 검사를 요구하지 않는다.",
    "계약상대자는 언제든지 일방적으로 계약을 해지할 수 있으며 그에 따른 손해는 발주기관이 부담한다.",
    "{o}의 추가 비용은 사유를 불문하고 발주기관이 전액 부담하며 별도 협의 없이 청구할 수 있다.",
    "{o}에 관한 지식재산권은 모두 계약상대자에게 귀속되고 발주기관은 사용할 수 없다.",
    "하도급은 신고 없이 제한 없이 허용하며 발주기관은 하도급 내용을 확인하지 않는다.",
    "대금은 검사 전에 전액 선지급하며 발주기관은 반환 청구권을 포기한다.",
    "계약 이행 중 알게 된 사항의 누설로 인한 책임을 계약상대자는 지지 않는다.",
    "하자 보수 책임은 인도 후 즉시 소멸하며 발주기관은 보수를 요구할 수 없다.",
    "발주기관은 계약상대자가 청구하는 금액을 거절할 수 없으며 거절하면 위약금을 지급한다.",
    "분쟁이 생기면 계약상대자가 정하는 중재 기관에서만 해결하고 발주기관은 따른다.",
    "발주기관의 검사는 생략하고 계약상대자의 시험 결과서로 대신한다.",
    "견적 금액은 사후 정산 없이 그대로 확정하고 증빙 자료는 받지 않는다.",
]
D_TPL = [
    "납기는 자재 수급 사정에 따라 계약상대자가 연장할 수 있다.",
    "납품 일정은 가능한 한 지키도록 노력하며 지연되어도 사유를 보고하지 않는다.",
    "공급 지연이나 선행 업체 사정도 불가항력으로 보아 납기를 자동 연장한다.",
    "착수일은 선행 업체의 납품이 끝난 뒤 별도로 정하며 완료 기한은 정하지 않는다.",
    "납기일 이후 1회에 한해 기한 없이 자동 연장되는 것으로 한다.",
    "납기는 계약상대자의 생산 일정이 확정되는 대로 통보한다.",
    "납품 시점은 계약상대자 사정이 허락하는 때로 하고 날짜는 따로 정하지 않는다.",
    "자재가 늦게 들어오면 완료 일정은 그만큼 자동으로 늦춰진다.",
    "기한 내에 인도하지 못해도 계약상대자에게 책임을 묻지 않는다.",
    "일정이 늦어져도 계약상대자는 새 계획서를 낼 필요가 없다.",
]
# 학습에 쓰지 않는 '처음 보는 표현'(모델이 문장 틀을 외웠는지 확인하는 용도)
HOLD = {
    "정상": ["검사에 불합격한 경우 계약상대자는 지정 기한 안에 재납품하며 재검사 비용은 상대자가 부담한다.",
           "계약상대자가 법령을 위반하여 계약을 이행하지 못하면 발주기관은 서면 통지 후 계약을 해지할 수 있다.",
           "납기가 지연될 우려가 있으면 상대자는 즉시 사유와 만회 계획을 서면으로 보고해야 한다.",
           "하자담보 책임 기간은 인수일부터 기산하며 기간 중 발생한 결함은 상대자가 책임진다."],
    "독소": ["발주기관은 계약상대자가 요구하는 대금을 이유 없이 거절하지 못하며 거절 시 위약금을 낸다.",
           "계약 내용에 관한 분쟁은 계약상대자가 지정하는 기관에서만 다루며 발주기관은 이에 따른다.",
           "발주기관의 검수 권한은 포기하고 계약상대자의 자체 시험 성적서로 갈음한다.",
           "계약상대자가 제출한 견적은 사후 정산 없이 확정 단가로 하며 증빙은 요구하지 않는다."],
    "납기위험": ["납품 시기는 상대자의 형편이 허락하는 시점으로 하고 별도 날짜는 정하지 않는다.",
             "원자재 확보가 늦어지면 완료 일자는 자동으로 뒤로 미뤄지는 것으로 본다.",
             "인도 기한은 참고용이며 지키지 못해도 책임을 묻지 않는다.",
             "일정 지연이 있어도 상대자는 변경 계획을 제출할 의무가 없다."],
}
SUFFIX = ["", "", "", " 세부 사항은 시행 세칙에 따른다.", " 이 조항은 계약 전체에 적용한다.", " 필요한 경우 양 당사자가 협의한다.", " 관련 서식은 별지를 따른다."]
SWAP = [("발주기관", "계약담당관"), ("계약상대자", "업체")]


def make_clause(rng, cls, hold=False, idx=None):
    """조항 한 문장 생성. 표현 일부는 동의어로 바꾸고 접미 문구를 무작위로 붙인다(idx를 주면 그 번호의 문장 틀을 사용)."""
    if idx is not None:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[idx].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    elif hold:
        s = HOLD[cls][rng.integers(len(HOLD[cls]))]
    else:
        tpl = {"정상": N_TPL, "독소": T_TPL, "납기위험": D_TPL}[cls]
        s = tpl[rng.integers(len(tpl))].format(p=rng.choice(["1.0", "1.5", "2.0"]), d=int(rng.choice([7, 10, 14, 30])), o=rng.choice(OBJ))
    if rng.random() < 0.3:
        a, b = SWAP[rng.integers(2)]
        s = s.replace(a, b)
    return s + SUFFIX[rng.integers(len(SUFFIX))]


def make_contracts(n=200, seed=7):
    """가상 계약 n건(속성 16개 + 조항 6개 + 규정 위반 정답)을 만든다."""
    rng = np.random.default_rng(seed)
    sl = rng.beta(1.2, 3.2, n)                                  # 계약서 작성이 허술한 정도(잠재 변수)
    ctype = rng.choice(TYPES, n, p=[.5, .3, .2])
    amount = np.clip(np.exp(rng.normal(np.log(220), 1.1, n)), 5, 5000).round(0)
    method = np.where(rng.random(n) < .06 + .3 * sl, "수의계약", rng.choice(METHODS[:3], n, p=[.5, .3, .2]))
    period = np.array([rng.integers(60, 300) if t == "물품" else rng.integers(90, 400) if t == "용역" else rng.integers(180, 720) for t in ctype])
    delivery = (period * rng.uniform(.5, 1.0, n)).astype(int)
    bad_logic = rng.random(n) < .015 * (1 + 3 * sl)
    delivery = np.where(bad_logic, period + rng.integers(10, 60, n), delivery)
    pen = np.where(rng.random(n) < .22 * sl, rng.choice([.3, .5, .8], n), rng.choice([1.0, 1.5, 2.0, 2.5], n))
    adv = np.where(rng.random(n) < .22 * sl, rng.choice([50, 60, 70], n), rng.choice([0, 10, 20, 30, 40], n))
    bond = np.where(rng.random(n) < .25 * sl + .5 * (adv > 40), rng.choice([0, 5], n), rng.choice([10, 10, 15, 20], n))
    war = np.array([rng.choice([3, 6]) if (rng.random() < .25 * s and t == "물품") else rng.choice([12, 24, 36]) if t == "물품" else rng.choice([0, 6, 12]) if t == "용역" else rng.choice([24, 36, 60]) for t, s in zip(ctype, sl)])
    insp = np.where(rng.random(n) < .3 * sl, rng.choice([21, 30], n), rng.choice([5, 7, 10, 14], n))
    sub = np.where(rng.random(n) < .15 * sl, rng.choice([60, 70, 80], n), rng.choice([0, 10, 20, 30, 40, 50], n))
    padj = rng.random(n) < .75 - .4 * sl
    term = rng.random(n) > .015 * (1 + 6 * sl)
    sec = rng.random(n) > .015 * (1 + 6 * sl)
    ip = rng.random(n) > .03 * (1 + 4 * sl)
    clauses, cls_true = [], []
    for i in range(n):
        pt, pd_ = .01 + .08 * sl[i], .02 + .08 * sl[i]
        cl = [rng.choice(CLASSES, p=[1 - pt - pd_, pt, pd_]) for _ in range(6)]
        cls_true.append(cl); clauses.append([make_clause(rng, c) for c in cl])
    df = pd.DataFrame({"contract_id": [f"C-{i + 1:04d}" for i in range(n)], "ctype": ctype, "method": method, "amount_m": amount,
                       "period_days": period, "delivery_days": delivery, "penalty_permil": pen, "advance_pct": adv, "perf_bond_pct": bond,
                       "warranty_months": war, "inspection_days": insp, "subcontract_pct": sub, "price_adjust": padj,
                       "termination": term, "security": sec, "ip_clause": ip})
    hard = ((pen < 1.0) | ((adv > 40) & (bond < 10)) | ((amount >= 300) & (method == "수의계약")) | ((ctype == "물품") & (war < 12))
            | ~term | ~sec | (sub > 50) | (delivery > period))
    toxic = np.array([c.count("독소") for c in cls_true])
    delay = np.array([c.count("납기위험") for c in cls_true])
    split = (method == "수의계약") & (amount >= 250) & (amount < 300) & (rng.random(n) < .6)      # 계약 쪼개기 의심(임계값 바로 아래)
    df["clauses"], df["clause_cls"] = clauses, cls_true
    df["n_toxic"], df["n_delay"] = toxic, delay
    viol = hard | (toxic > 0) | split | (delay >= 2)
    waived = hard & (toxic == 0) & (rng.random(n) < .12)                  # 계약관이 예외 사유를 인정해 위반으로 보지 않은 건(자료의 현실적 잡음)
    hidden = ~viol & (rng.random(n) < .03 + .08 * sl)                       # 속성·문구에 드러나지 않는 위반(별첨 합의 등)
    df["violation"] = (viol & ~waived) | hidden
    df["vtype"] = np.where(hidden, "기타", np.where(hard, "속성위반", np.where(toxic > 0, "독소조항", np.where(split | (delay >= 2), "기타", "-"))))
    return df

# ---- 계약 규정 준수 판별 룰 설정(YAML 형식 문자열: 규정이 바뀌면 코드가 아니라 이 설정만 고친다) ----
RULES_YAML = """
- id: R01
  name: 지체상금률 1.0/1000 미만
  field: penalty_permil
  op: "<"
  value: 1.0
  weight: 3
- id: R02
  name: 선금 비율 40% 초과
  field: advance_pct
  op: ">"
  value: 40
  weight: 2
- id: R03
  name: 선금 40% 초과인데 이행보증 10% 미만
  field: adv_no_bond
  op: "=="
  value: true
  weight: 3
- id: R04
  name: 300백만원 이상 수의계약
  field: sole_amount
  op: ">="
  value: 300
  weight: 3
- id: R05
  name: 500백만원 이상 이행보증 10% 미만
  field: bond_gap
  op: "=="
  value: true
  weight: 2
- id: R06
  name: 물품 하자보증 12개월 미만
  field: warranty_gap
  op: "=="
  value: true
  weight: 2
- id: R07
  name: 해지 조항 없음
  field: termination
  op: "=="
  value: false
  weight: 2
- id: R08
  name: 보안 조항 없음
  field: security
  op: "=="
  value: false
  weight: 3
- id: R09
  name: 지식재산 조항 없음(물품·용역)
  field: ip_missing
  op: "=="
  value: true
  weight: 1
- id: R10
  name: 하도급 비율 50% 초과
  field: subcontract_pct
  op: ">"
  value: 50
  weight: 3
- id: R11
  name: 검사 기간 14일 초과
  field: inspection_days
  op: ">"
  value: 14
  weight: 1
- id: R12
  name: 납품 기한이 계약 기간 초과
  field: logic_err
  op: "=="
  value: true
  weight: 3
- id: R13
  name: 180일 이상 계약에 물가변동 조항 없음
  field: no_padj_long
  op: "=="
  value: true
  weight: 1
- id: R14
  name: 1000백만원 이상 비경쟁 계약
  field: noncomp_big
  op: "=="
  value: true
  weight: 2
- id: R15
  name: 계약 기간 365일 초과
  field: period_days
  op: ">"
  value: 365
  weight: 1
- id: R16
  name: 불공정·납기 위험 키워드 포함
  field: kw_hits
  op: ">="
  value: 1
  weight: 3
"""
GRADE_CUT = {"고위험": 6, "주의": 3}          # 가중치 합 기준 1차 판별 등급(필요 시 설정만 바꾼다)
# [교육용가상자료] 불공정 조항·납기 지체 위험 문구 사전(실제 규정 조문이 아님)
KEYWORDS = {"독소": ["면제하며", "일방적으로", "포기", "지지 않는다", "즉시 소멸", "제한 없이", "신고 없이", "전액 부담", "별도 협의 없이", "검사 전에", "검사에 합격한 것으로", "사용할 수 없다", "증빙은 요구하지 않는다", "지정하는 기관"],
            "납기위험": ["자동 연장", "기한 없이", "정하지 않는다", "노력하며", "연장할 수 있다", "통보한다", "책임을 묻지 않는다", "의무가 없다"]}


def parse_rules(text):
    """간이 YAML 파서(목록 + key: value). 숫자·true/false·따옴표 문자열을 해석한다."""
    # [TODO : 학생 실습 5] 계약 위험 점수 산출 및 규정 위반 가능성 신뢰도 지수 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
OPS = {"==": lambda a, b: a == b, "!=": lambda a, b: a != b, ">": lambda a, b: a > b, ">=": lambda a, b: a >= b,
       "<": lambda a, b: a < b, "<=": lambda a, b: a <= b}


def kw_count(clauses):
    """조항 문장 목록에서 독소·납기 위험 키워드가 들어 있는 조항 수."""
    # [TODO : 학생 실습 5] 계약 위험 점수 산출 및 규정 위반 가능성 신뢰도 지수 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def derive(df):
    """룰에 쓰는 파생 필드(선금+보증 부족, 수의계약 고액, 논리 오류, 키워드 적중 등)를 만든다."""
    d = df.copy()
    d["adv_no_bond"] = (d["advance_pct"] > 40) & (d["perf_bond_pct"] < 10)
    d["sole_amount"] = np.where(d["method"] == "수의계약", d["amount_m"], 0.0)       # 수의계약일 때만 금액(한도는 YAML에서 조정)
    d["bond_gap"] = (d["amount_m"] >= 500) & (d["perf_bond_pct"] < 10)
    d["warranty_gap"] = (d["ctype"] == "물품") & (d["warranty_months"] < 12)
    d["ip_missing"] = (~d["ip_clause"]) & (d["ctype"] != "공사")
    d["logic_err"] = d["delivery_days"] > d["period_days"]
    d["no_padj_long"] = (d["period_days"] >= 180) & (~d["price_adjust"])
    d["noncomp_big"] = d["method"].isin(["지명경쟁", "수의계약"]) & (d["amount_m"] >= 1000)
    d["kw_hits"] = [kw_count(c) for c in d["clauses"]] if "clauses" in d else 0
    return d


def apply_rules(rec, rules):
    """레코드 1건(dict)에 룰을 적용해 (점수, 등급, 적중 룰 id 목록)을 돌려준다. 값이 없으면 해당 룰은 건너뛴다."""
    hits = []
    for r in rules:
        v = rec.get(r["field"])
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        if OPS[r["op"]](v, r["value"]):
            hits.append(r)
    score = sum(r["weight"] for r in hits)
    grade = "고위험" if score >= GRADE_CUT["고위험"] else ("주의" if score >= GRADE_CUT["주의"] else "관심")
    return score, grade, [r["id"] for r in hits]

SIMPLE_KW = {"독소": ["면제", "일방", "포기", "전액", "누설", "하도급", "선지급"], "납기위험": ["연장", "자동", "노력", "통보", "기한"]}

# ---- 조항 문장 분류(정상 / 독소 / 납기위험): 글자 n-gram TF-IDF + 로지스틱 회귀 ----
def clause_set(seed, n, hold=False, mix=(.55, .25, .20)):
    """조항 문장 n개와 정답 분류를 만든다. hold=True면 학습에 쓰지 않은 '처음 보는 표현'으로 만든다."""
    rng = np.random.default_rng(seed)
    y = rng.choice(CLASSES, n, p=list(mix))
    return [make_clause(rng, c, hold) for c in y], list(y)


def fit_clause_model(seed=100, n=1200):
    """학습용 조항 문장으로 분류 모델을 학습한다."""
    X, y = clause_set(seed, n)
    return make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True), LogisticRegression(C=20, max_iter=2000)).fit(X, y)


def keyword_pred(s):
    """비교용 기준선: 단순 키워드(부정 표현·문맥을 보지 않음) 적중으로만 분류."""
    if any(k in s for k in SIMPLE_KW["독소"]):
        return "독소"
    return "납기위험" if any(k in s for k in SIMPLE_KW["납기위험"]) else "정상"

RULES = parse_rules(RULES_YAML)
RID = [r["id"] for r in RULES]
WARN = {"고위험": 60, "주의": 30}                       # 위험 점수 등급 기준(점수 = 위반 확률 x 100)


def featurize(con, cm):
    """룰 16종 적중 여부 + 조항 모델이 찾은 독소·납기위험 조항 수 + 금액(로그)."""
    d = derive(con)
    H = np.array([[r in apply_rules(x, RULES)[2] for r in RID] for x in d.to_dict("records")], float)
    flat = [c for cl in con["clauses"] for c in cl]
    pred = np.array(cm.predict(flat)).reshape(len(con), 6)
    nt, nd = (pred == "독소").sum(1), (pred == "납기위험").sum(1)
    return np.column_stack([H, nt, nd, np.log1p(con["amount_m"].to_numpy())]), H, nt, nd


FEAT_NAMES = [f"{r['id']} {r['name']}" for r in RULES] + ["독소 조항 수(모델)", "납기위험 문구 수(모델)", "금액(로그)"]
cm = fit_clause_model()
train, test = make_contracts(800, seed=11), make_contracts(300, seed=7)
Xtr, _, _, _ = featurize(train, cm)
Xte, Hte, nt, nd = featurize(test, cm)
lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=1000)).fit(Xtr, train["violation"].astype(int))
y = test["violation"].astype(int).to_numpy()
p = lr.predict_proba(Xte)[:, 1]
score = p * 100
rule_flag = np.array([apply_rules(x, RULES)[1] != "관심" for x in derive(test).to_dict("records")])
agree = (rule_flag == (p >= 0.5)).astype(float)
conf = np.clip(0.6 * np.abs(2 * p - 1) + 0.4 * agree, 0, 1) * 100                      # 신뢰도 지수: 확률의 뚜렷함 + 룰·모델 일치
level = np.where(score >= WARN["고위험"], "고위험", np.where(score >= WARN["주의"], "주의", "관심"))
T = test.assign(score=score, conf=conf, level=level, y=y)

# ===== 화면 1. 입력: 입력 요약과 검증 =====
miss = int(test[["amount_m", "penalty_permil", "advance_pct"]].isna().sum().sum())
print("[화면 1. 입력] [교육용가상자료]")
print(f"  검증용 계약 {len(test)}건(조항 {len(test) * 6}개) / 학습 계약 {len(train)}건 / 입력 지표 {len(FEAT_NAMES)}개 / 필수값 결측 {miss}건")
print(f"  수의계약 {int((test['method'] == '수의계약').sum())}건, 평균 금액 {test['amount_m'].mean():.0f}백만원, 보안 조항 없음 {int((~test['security']).sum())}건")

# ===== 화면 2. 분석: 점수 분포와 성능 =====
auc = roc_auc_score(y, score)
print("\n[화면 2. 분석] 위반 위험 점수와 모델 성능")
print(f"  ROC-AUC {auc:.3f}, Brier {brier_score_loss(y, p):.3f} (실제 위반 비율 {y.mean():.0%}), 룰 점수 단독 AUC {roc_auc_score(y, Hte @ np.array([r['weight'] for r in RULES])):.3f}")
print(T.groupby("level").agg(계약수=("contract_id", "size"), 평균점수=("score", "mean"), 실제위반율=("y", "mean")).reindex(["고위험", "주의", "관심"]).round(2).to_string())
coef = pd.Series(lr[-1].coef_[0], index=FEAT_NAMES).sort_values(ascending=False)
print("  위험 상승 요인(회귀 계수 상위 4):", ", ".join(f"{k.split(' ')[0]} {v:+.2f}" for k, v in coef.head(4).items()))
print("  [신뢰도 지수와 실제 정확도] (판정 = 점수 50 이상을 위반 가능으로 봄)")
bins = pd.cut(T["conf"], [0, 50, 75, 90, 100.1], labels=["50 미만", "50~75", "75~90", "90 이상"])
ok = ((p >= 0.5).astype(int) == y)
cal = pd.DataFrame({"bin": bins, "ok": ok}).groupby("bin", observed=True).agg(건수=("ok", "size"), 실제정확도=("ok", "mean")).round(2)
print(cal.to_string())

# ===== 화면 3. 검토: 고위험 계약을 담당자가 확인 =====
rev = T[T["level"] == "고위험"].sort_values("score", ascending=False)
contrib = (lr[0].transform(Xte) * lr[-1].coef_[0])
BADGE = {"고위험": "[검토 필수]", "주의": "[확인 권고]", "관심": "[통과 후보]"}
print(f"\n[화면 3. 검토] 고위험 {len(rev)}건 중 상위 5건 (담당자가 근거를 확인하고 조치 여부를 결정)")
for i, r in rev.head(5).iterrows():
    top = [FEAT_NAMES[j].split(" ")[0] + ":" + FEAT_NAMES[j].split(" ", 1)[1][:10] for j in np.argsort(-contrib[i])[:2] if contrib[i][j] > 0]
    print(f"  {BADGE[r['level']]} {r['contract_id']} {r['ctype']} {r['amount_m']:.0f}백만원 | 위험 {r['score']:.0f}점 신뢰도 {r['conf']:.0f} | 요인 {', '.join(top) or '-'} | 상태 검토대기")

# ===== 화면 4. 결과: 요약, 뱃지 필터, 내보내기 =====
print("\n[화면 4. 결과]")
cnt = T["level"].value_counts()
print(f"  {BADGE['고위험']} {cnt.get('고위험', 0)}건, {BADGE['주의']} {cnt.get('주의', 0)}건, {BADGE['관심']} {cnt.get('관심', 0)}건")
f = T[(T["level"] == "주의") & (T["conf"] < 75)]
print(f"  필터 예시: 등급 '주의' 이면서 신뢰도 75 미만 -> {len(f)}건(사람이 먼저 확인할 경계 사례)")
print(f"  고위험 등급의 실제 위반 {int(rev['y'].sum())}건({rev['y'].mean():.0%}) vs 관심 등급 {T[T['level'] == '관심']['y'].mean():.0%}")
print("  내보내기 항목: 계약번호, 위험 점수, 신뢰도, 등급 뱃지, 요인, 검토 상태(검토대기)")
print("화면 4종(입력-분석-검토-결과) 구현: 4/4 (목표 4종: 달성)")
print("※ 점수와 신뢰도는 우선순위를 정하는 참고값이며, 규정 위반 여부의 판단과 조치는 계약관·법무 담당자가 한다.")

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0, 0].bar(["sole-source", "no security", "no termination"], [(test["method"] == "수의계약").sum(), (~test["security"]).sum(), (~test["termination"]).sum()]); ax[0, 0].set_title("Screen 1: input summary (count)")
ax[0, 1].hist([T[T["y"] == 0]["score"], T[T["y"] == 1]["score"]], bins=15, label=["no violation", "violation"], stacked=True); ax[0, 1].legend(); ax[0, 1].set_title("Screen 2: risk score distribution")
ax[1, 0].scatter(T["score"], T["conf"], c=T["y"], cmap="coolwarm", s=14); ax[1, 0].set_xlabel("risk score"); ax[1, 0].set_ylabel("confidence index"); ax[1, 0].set_title("Screen 3: review map (color = actual)")
ax[1, 1].bar(["high", "watch", "low"], [cnt.get("고위험", 0), cnt.get("주의", 0), cnt.get("관심", 0)], color=["tab:red", "gold", "tab:green"]); ax[1, 1].set_title("Screen 4: result by grade")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[화면 1. 입력] [교육용가상자료]
  검증용 계약 300건(조항 1800개) / 학습 계약 800건 / 입력 지표 19개 / 필수값 결측 0건
  수의계약 38건, 평균 금액 381백만원, 보안 조항 없음 10건

[화면 2. 분석] 위반 위험 점수와 모델 성능
  ROC-AUC 0.954, Brier 0.062 (실제 위반 비율 41%), 룰 점수 단독 AUC 0.889
       계약수   평균점수  실제위반율
level                   
고위험    112  90.48   0.94
주의       8  48.49   1.00
관심     180  10.12   0.05
  위험 상승 요인(회귀 계수 상위 4): 독소 +2.62, R07 +1.04, R01 +0.98, R04 +0.76
  [신뢰도 지수와 실제 정확도] (판정 = 점수 50 이상을 위반 가능으로 봄)
        건수  실제정확도
bin              
50 미만   38   0.84
50~75   31   0.87
75~90   67   0.97
90 이상  164   0.95

[화면 3. 검토] 고위험 112건 중 상위 5건 (담당자가 근거를 확인하고 조치 여부를 결정)
  [검토 필수] C-0222 물품 162백만원 | 위험 100점 신뢰도 100 | 요인 독소:조항 수(모델), R06:물품 하자보증 12 | 상태 검토대기
  [검토 필수] C-0028 물품 590백만원 | 위험 100점 신뢰도 100 | 요인 독소:조항 수(모델), R08:보안 조항 없음 | 상태 검토대기
  [검토 필수] C-0266 물품 68백만원 | 위험 100점 신뢰도 100 | 요인 독소:조항 수(모델), R08:보안 조항 없음 | 상태 검토대기
  [검토 필수] C-0037 물품 62백만원 | 위험 100점 신뢰도 100 | 요인 독소:조항 수(모델), R10:하도급 비율 50% | 상태 검토대기
  [검토 필수] C-0221 용역 101백만원 | 위험 100점 신뢰도 100 | 요인 독소:조항 수(모델), R10:하도급 비율 50% | 상태 검토대기

[화면 4. 결과]
  [검토 필수] 112건, [확인 권고] 8건, [통과 후보] 180건
  필터 예시: 등급 '주의' 이면서 신뢰도 75 미만 -> 8건(사람이 먼저 확인할 경계 사례)
  고위험 등급의 실제 위반 105건(94%) vs 관심 등급 5%
  내보내기 항목: 계약번호, 위험 점수, 신뢰도, 등급 뱃지, 요인, 검토 상태(검토대기)
화면 4종(입력-분석-검토-결과) 구현: 4/4 (목표 4종: 달성)
※ 점수와 신뢰도는 우선순위를 정하는 참고값이며, 규정 위반 여부의 판단과 조치는 계약관·법무 담당자가 한다.
```
</details>
